# Notebook 03 — Data Integration & Feature Engineering

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 5 — Integration & Feature Engineering  

## Objectives
- Integrate staged tables into a robust enterprise **Star Schema** (Facts & Dimensions)
- Build `fact_orders` at order grain with operational KPIs, payment summaries, and review metrics
- Build `fact_order_items` at item grain enriched with seller/product dimensions and Haversine shipment distance
- Engineer delivery cycle metrics: approval hours, handling days, transit days, total delivery days, and delay days
- Create classification bands: delivery status, delay bands, review groups, and distance bands
- Construct customer summary (`dim_customers`) and seller scorecard (`dim_sellers`) dimensions
- Build standard corporate calendar dimension (`dim_date`)

---
## 0. Setup & Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

STAGING_PATH = '../data/staging/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

print('Setup complete. Staging path:', STAGING_PATH)

---
## 1. Load Clean Staged Data

In [ ]:
stg_orders = pd.read_csv(os.path.join(STAGING_PATH, 'stg_orders.csv'))
stg_items = pd.read_csv(os.path.join(STAGING_PATH, 'stg_order_items.csv'))
stg_customers = pd.read_csv(os.path.join(STAGING_PATH, 'stg_customers.csv'))
stg_products = pd.read_csv(os.path.join(STAGING_PATH, 'stg_products.csv'))
stg_sellers = pd.read_csv(os.path.join(STAGING_PATH, 'stg_sellers.csv'))
stg_payments_agg = pd.read_csv(os.path.join(STAGING_PATH, 'stg_payments_order_agg.csv'))
stg_reviews_agg = pd.read_csv(os.path.join(STAGING_PATH, 'stg_reviews_order_agg.csv'))
stg_geo = pd.read_csv(os.path.join(STAGING_PATH, 'stg_geolocation_zip.csv'))

date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in date_cols:
    stg_orders[c] = pd.to_datetime(stg_orders[c], errors='coerce')

print('All staged files loaded successfully.')

---
## 2. Aggregating Items & Constructing `fact_orders`

**Core Metrics Engineered:**
- `approval_hours`: $\text{Order Approved} - \text{Order Purchased}$ (in hours)
- `handling_days`: $\text{Carrier Handover} - \text{Order Approved}$ (in days)
- `transit_days`: $\text{Delivered to Customer} - \text{Carrier Handover}$ (in days)
- `delivery_days`: Total end-to-end fulfillment time
- `delay_days`: $\text{Actual Delivery} - \text{Estimated Delivery}$ (positive = late, negative/zero = early or on-time)
- `late_delivery_flag` & `severe_delay_flag` (late by > 7 days)

In [ ]:
items_agg = stg_items.groupby('order_id').agg(
    item_count=('order_item_id', 'count'),
    unique_product_count=('product_id', 'nunique'),
    unique_seller_count=('seller_id', 'nunique'),
    gmv=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    total_order_value=('item_total_value', 'sum'),
    average_item_price=('price', 'mean'),
    maximum_item_price=('price', 'max'),
    average_freight_to_price_ratio=('freight_to_price_ratio', 'mean')
).reset_index()

fact_orders = stg_orders.merge(stg_customers, on='customer_id', how='left')
fact_orders = fact_orders.merge(items_agg, on='order_id', how='left')
fact_orders = fact_orders.merge(stg_payments_agg, on='order_id', how='left')
fact_orders = fact_orders.merge(stg_reviews_agg, on='order_id', how='left')

for col in ['item_count', 'unique_product_count', 'unique_seller_count', 'gmv', 'freight_value', 'total_order_value']:
    fact_orders[col] = fact_orders[col].fillna(0)

# Delivery Cycle Calculations
fact_orders['approval_hours'] = ((fact_orders['order_approved_at'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 3600.0).round(2)
fact_orders['handling_days'] = ((fact_orders['order_delivered_carrier_date'] - fact_orders['order_approved_at']).dt.total_seconds() / 86400.0).round(2)
fact_orders['transit_days'] = ((fact_orders['order_delivered_customer_date'] - fact_orders['order_delivered_carrier_date']).dt.total_seconds() / 86400.0).round(2)
fact_orders['delivery_days'] = ((fact_orders['order_delivered_customer_date'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400.0).round(2)
fact_orders['promised_delivery_days'] = ((fact_orders['order_estimated_delivery_date'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400.0).round(2)
fact_orders['delay_days'] = ((fact_orders['order_delivered_customer_date'] - fact_orders['order_estimated_delivery_date']).dt.total_seconds() / 86400.0).round(2)

# Delivery Classification
def get_delivery_status(row):
    if row['order_status'] != 'delivered' or pd.isna(row['order_delivered_customer_date']):
        return 'Not Delivered'
    elif row['delay_days'] <= 0:
        return 'Early / On Time'
    else:
        return 'Late'

fact_orders['delivery_status'] = fact_orders.apply(get_delivery_status, axis=1)

is_delivered = (fact_orders['order_status'] == 'delivered') & fact_orders['order_delivered_customer_date'].notna()
fact_orders['late_delivery_flag'] = np.where(is_delivered & (fact_orders['delay_days'] > 0), 1, 0)
fact_orders['severe_delay_flag'] = np.where(is_delivered & (fact_orders['delay_days'] > 7), 1, 0)

def get_delay_band(row):
    if row['order_status'] != 'delivered' or pd.isna(row['order_delivered_customer_date']):
        return 'Not Delivered'
    elif row['delay_days'] <= 0: return 'Early or On Time'
    elif row['delay_days'] <= 3: return '1–3 Days Late'
    elif row['delay_days'] <= 7: return '4–7 Days Late'
    else: return '8+ Days Late'

fact_orders['delay_band'] = fact_orders.apply(get_delay_band, axis=1)

# Review Features
def get_review_group(score):
    if pd.isna(score): return 'No Review'
    elif score >= 4: return 'Positive (4-5)'
    elif score == 3: return 'Neutral (3)'
    else: return 'Negative (1-2)'

fact_orders['review_group'] = fact_orders['review_score_avg'].apply(get_review_group)
fact_orders['low_review_flag'] = np.where(fact_orders['review_score_avg'] <= 2, 1, 0)
fact_orders['high_review_flag'] = np.where(fact_orders['review_score_avg'] >= 4, 1, 0)

fact_orders.to_csv(os.path.join(PROCESSED_PATH, 'fact_orders.csv'), index=False)
print(f"fact_orders: {fact_orders.shape[0]:,} rows x {fact_orders.shape[1]} cols")
fact_orders[['order_id', 'order_status', 'gmv', 'freight_value', 'delivery_days', 'delay_days', 'delivery_status', 'review_group']].head(3)

---
## 3. Constructing `fact_order_items` with Haversine Distance

Calculates great-circle distance between customer and seller geographic coordinates.

In [ ]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2.0) ** 2 +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2.0) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

fact_items = stg_items.merge(
    fact_orders[['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date', 'delivery_days', 'delay_days', 'delivery_status',
                 'late_delivery_flag', 'customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
                 'customer_city', 'customer_state']],
    on='order_id', how='left'
)

fact_items = fact_items.merge(
    stg_products[['product_id', 'product_category_name', 'product_category_name_english',
                  'product_volume_cm3', 'product_size_band', 'product_weight_g', 'product_weight_band']],
    on='product_id', how='left'
)

fact_items = fact_items.merge(
    stg_sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
    on='seller_id', how='left'
)

fact_items = fact_items.merge(
    stg_geo.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'latitude': 'cust_lat', 'longitude': 'cust_lng'})[['customer_zip_code_prefix', 'cust_lat', 'cust_lng']],
    on='customer_zip_code_prefix', how='left'
)

fact_items = fact_items.merge(
    stg_geo.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix', 'latitude': 'seller_lat', 'longitude': 'seller_lng'})[['seller_zip_code_prefix', 'seller_lat', 'seller_lng']],
    on='seller_zip_code_prefix', how='left'
)

fact_items['distance_km'] = haversine_vectorized(
    fact_items['seller_lat'], fact_items['seller_lng'],
    fact_items['cust_lat'], fact_items['cust_lng']
).round(1)

def get_distance_band(d):
    if pd.isna(d): return 'Unknown'
    elif d <= 100: return '0–100 km'
    elif d <= 500: return '101–500 km'
    elif d <= 1000: return '501–1,000 km'
    elif d <= 2000: return '1,001–2,000 km'
    else: return '2,000+ km'

fact_items['distance_band'] = fact_items['distance_km'].apply(get_distance_band)
fact_items = fact_items.drop(columns=['cust_lat', 'cust_lng', 'seller_lat', 'seller_lng'])

fact_items.to_csv(os.path.join(PROCESSED_PATH, 'fact_order_items.csv'), index=False)
print(f"fact_order_items: {fact_items.shape[0]:,} rows x {fact_items.shape[1]} cols")
fact_items[['order_id', 'order_item_id', 'price', 'freight_value', 'distance_km', 'distance_band']].head(3)

---
## 4. Customer Summary Dimension (`dim_customers`)

Aggregates order history by `customer_unique_id` to measure lifetime value, repeat frequency, and satisfaction.

In [ ]:
cust_summary = fact_orders.groupby('customer_unique_id').agg(
    first_purchase_date=('order_purchase_timestamp', 'min'),
    last_purchase_date=('order_purchase_timestamp', 'max'),
    total_orders=('order_id', 'count'),
    total_gmv=('gmv', 'sum'),
    total_freight=('freight_value', 'sum'),
    avg_review_score=('review_score_avg', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    late_order_count=('late_delivery_flag', 'sum'),
    primary_state=('customer_state', 'first'),
    primary_city=('customer_city', 'first'),
    primary_zip_prefix=('customer_zip_code_prefix', 'first')
).reset_index()

cust_summary['average_order_value'] = (cust_summary['total_gmv'] / cust_summary['total_orders']).round(2)
cust_summary['repeat_customer_flag'] = (cust_summary['total_orders'] > 1).astype(int)
cust_summary['avg_review_score'] = cust_summary['avg_review_score'].round(2)
cust_summary['avg_delivery_days'] = cust_summary['avg_delivery_days'].round(2)

cust_summary.to_csv(os.path.join(PROCESSED_PATH, 'dim_customers.csv'), index=False)
print(f"dim_customers: {cust_summary.shape[0]:,} unique customers")
cust_summary.head(3)

---
## 5. Seller Scorecard Dimension (`dim_sellers`)

Calculates fulfillment operational KPIs for every seller: GMV, items sold, on-time delivery rate, and average review score.

In [ ]:
seller_perf = fact_items.groupby('seller_id').agg(
    total_orders=('order_id', 'nunique'),
    items_sold=('order_item_id', 'count'),
    total_gmv=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    avg_delay_days=('delay_days', 'mean'),
    states_served_count=('customer_state', 'nunique'),
    categories_sold_count=('product_category_name_english', 'nunique'),
    seller_city=('seller_city', 'first'),
    seller_state=('seller_state', 'first'),
    seller_zip_code_prefix=('seller_zip_code_prefix', 'first')
).reset_index()

delivered_items = fact_items[fact_items['order_status'] == 'delivered'].copy()
seller_deliv = delivered_items.groupby('seller_id').agg(
    delivered_items_count=('order_item_id', 'count'),
    late_items_count=('late_delivery_flag', 'sum')
).reset_index()

seller_perf = seller_perf.merge(seller_deliv, on='seller_id', how='left').fillna(0)
seller_perf['on_time_rate'] = np.where(
    seller_perf['delivered_items_count'] > 0,
    (1.0 - (seller_perf['late_items_count'] / seller_perf['delivered_items_count'])).clip(lower=0) * 100.0,
    100.0
).round(2)

seller_perf['late_delivery_rate'] = (100.0 - seller_perf['on_time_rate']).round(2)
seller_perf['freight_to_price_ratio'] = np.where(
    seller_perf['total_gmv'] > 0,
    (seller_perf['total_freight'] / seller_perf['total_gmv']).round(4),
    0.0
)

seller_rev = fact_items.groupby('seller_id')['order_id'].unique().reset_index()
order_rev_map = fact_orders.set_index('order_id')['review_score_avg'].to_dict()
seller_rev['avg_review_score'] = seller_rev['order_id'].apply(
    lambda order_list: np.nanmean([order_rev_map.get(oid, np.nan) for oid in order_list])
).round(2)

seller_perf = seller_perf.merge(seller_rev[['seller_id', 'avg_review_score']], on='seller_id', how='left')
seller_perf.to_csv(os.path.join(PROCESSED_PATH, 'dim_sellers.csv'), index=False)
print(f"dim_sellers: {seller_perf.shape[0]:,} sellers scored")
seller_perf.head(3)

---
## 6. Corporate Calendar Dimension (`dim_date`)

Builds a comprehensive date dimension from 2016-01-01 to 2018-12-31 to support time intelligence in Power BI.

In [ ]:
date_range = pd.date_range(start='2016-01-01', end='2018-12-31', freq='D')
dim_date = pd.DataFrame({'date': date_range})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['quarter_name'] = 'Q' + dim_date['quarter'].astype(str)
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['month_year'] = dim_date['date'].dt.strftime('%b %Y')
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_name'] = dim_date['date'].dt.strftime('%A')
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek + 1
dim_date['is_weekend'] = dim_date['day_of_week'].apply(lambda x: 1 if x in [6, 7] else 0)
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')

dim_date.to_csv(os.path.join(PROCESSED_PATH, 'dim_date.csv'), index=False)
print(f"dim_date: {dim_date.shape[0]:,} calendar days created")
dim_date.head(3)